# GemmaCode E2B — Phase 1 Baseline (01_baseline)

This notebook establishes the Phase 1 foundation for **GemmaCode E2B**:
- Validate the Google Colab environment for inference
- Authenticate with Hugging Face (gated Gemma 4 access)
- Load `google/gemma-4-E2B-it` using the verified Gemma 4 loading procedure
- Run a 5-task software-engineering baseline benchmark
- Save results locally and to Google Drive (without overwriting previous runs)

**Important constraints**
- Run this notebook in **Google Colab** (GPU runtime recommended).
- Do **not** download full model weights to a local laptop/PC.
- Do **not** hardcode tokens/secrets.
- Phase 1 is baseline validation only (no fine-tuning).


## A. Environment diagnostics
Collect runtime details and assess whether the environment is suitable for Gemma 4 E2B inference.


In [ ]:
import os
import platform
import sys
from pathlib import Path

def find_project_root() -> Path:
    """Locate GemmaCode-E2B root by expected marker files (works in Colab/local)."""
    markers = ('benchmarks/coding_tasks.jsonl', 'configs/model.yaml')
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    # Common Colab locations
    candidates.extend([
        Path('/content'),
        Path('/content/GemmaCode-E2B'),
        Path('/content/drive/MyDrive/GemmaCode-E2B'),
    ])
    # Also scan immediate children of /content for uploaded repos
    content = Path('/content')
    if content.exists():
        try:
            candidates.extend([p for p in content.iterdir() if p.is_dir()])
        except Exception:
            pass
    seen = set()
    for c in candidates:
        try:
            r = c.resolve()
        except Exception:
            continue
        if r in seen:
            continue
        seen.add(r)
        if all((r / m).exists() for m in markers):
            return r
    # Fallback: if running inside notebooks/
    if cwd.name == 'notebooks':
        return cwd.parent
    return cwd

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('cwd:', Path.cwd())
print('Tasks file exists:', (PROJECT_ROOT / 'benchmarks/coding_tasks.jsonl').exists())

try:
    from src.evaluation.env_diagnostics import collect_environment_report, assess_inference_suitability, print_environment_report
    report = collect_environment_report()
    assessment = assess_inference_suitability(report)
    print_environment_report(report, assessment)
except Exception as e:
    # Fallback inline diagnostics if project modules are unavailable
    print('Could not import project diagnostics module:', e)
    try:
        import psutil, torch
        import transformers
        print('python_version:', sys.version.split()[0])
        print('platform:', platform.platform())
        print('torch_version:', torch.__version__)
        print('transformers_version:', transformers.__version__)
        print('cuda_available:', torch.cuda.is_available())
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            print('gpu_name:', props.name)
            print('gpu_vram_gb:', round(props.total_memory / (1024**3), 2))
            print('gpu_compute_capability:', f'{props.major}.{props.minor}')
        print('system_ram_gb:', round(psutil.virtual_memory().total / (1024**3), 2))
    except Exception as inner:
        print('Fallback diagnostics unavailable:', inner)


## B. Dependency installation (baseline inference only)
Installs/verifies packages required for Gemma 4 baseline inference in Colab. Avoid unnecessary upgrades.


In [ ]:
# Run in Google Colab. Torch is typically preinstalled; upgrade/install only what is needed.
import sys

IN_COLAB = 'google.colab' in sys.modules
print('IN_COLAB:', IN_COLAB)

if IN_COLAB:
    # Official Gemma 4 docs indicate transformers>=5.10.1 for current APIs.
    !pip install -q --upgrade "transformers>=5.10.1" accelerate huggingface_hub safetensors pyyaml psutil
else:
    print('Skipping pip install (not running in Colab). Ensure baseline inference deps are installed in your environment if needed.')


In [ ]:
# Verify key package versions after installation
import importlib
for pkg in ['torch', 'transformers', 'accelerate', 'huggingface_hub', 'safetensors']:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, '__version__', 'unknown')
        print(f'{pkg}: {ver}')
    except Exception as e:
        print(f'{pkg}: NOT AVAILABLE ({e})')


## C. Hugging Face authentication (Colab Secrets)
Gemma models may be gated. Authenticate using a Hugging Face token stored in Colab Secrets (recommended key: `HF_TOKEN`).


In [ ]:
import os

HF_TOKEN = None
IN_COLAB = 'google.colab' in sys.modules

# Prefer Colab Secrets
if IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        if HF_TOKEN:
            print('Loaded HF_TOKEN from Colab Secrets.')
    except Exception as e:
        print('Colab Secrets not available or HF_TOKEN not set:', e)

# Fallback to environment variable (useful outside Colab if explicitly set by user)
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
    if HF_TOKEN:
        print('Loaded HF token from environment variable.')

if not HF_TOKEN:
    print('WARNING: No Hugging Face token found. Gated model access may fail.')
    print('In Colab: set Secret name HF_TOKEN and grant notebook access.')

# Optional login for hub APIs; transformers.from_pretrained can also receive token directly
try:
    from huggingface_hub import login
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=False)
        print('Hugging Face login successful.')
except Exception as e:
    print('huggingface_hub login skipped/failed:', e)


## D. Load the original model
Load `google/gemma-4-E2B-it` with the verified Gemma 4 API (`AutoProcessor` + `AutoModelForMultimodalLM`).
Model weights are downloaded by Hugging Face/Transformers in the Colab runtime (not to your local laptop).


In [ ]:
import os
import time
import torch
from pathlib import Path

MODEL_ID = 'google/gemma-4-E2B-it'

# Cache location: keep model downloads in Colab VM when running from Drive (faster, avoids Drive quota),
# otherwise keep caches inside the project workspace. Never download weights to a local laptop.
IN_COLAB = 'google.colab' in sys.modules
project_str = str(Path(PROJECT_ROOT).resolve())
on_drive = project_str.startswith('/content/drive')
if IN_COLAB and on_drive:
    cache_root = Path('/content/.cache/gemmacode')
    print('Project is on Drive; using Colab-local cache for model downloads:', cache_root)
else:
    cache_root = (PROJECT_ROOT / '.cache').resolve()
hf_home = cache_root / 'huggingface'
transformers_cache = cache_root / 'transformers'
for p in [cache_root, hf_home, transformers_cache]:
    p.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(hf_home))
os.environ.setdefault('TRANSFORMERS_CACHE', str(transformers_cache))
print('HF_HOME:', os.environ.get('HF_HOME'))
print('TRANSFORMERS_CACHE:', os.environ.get('TRANSFORMERS_CACHE'))

def _gpu_mem():
    if torch.cuda.is_available():
        return {
            'allocated_gb': round(torch.cuda.memory_allocated() / (1024**3), 3),
            'reserved_gb': round(torch.cuda.memory_reserved() / (1024**3), 3),
            'max_allocated_gb': round(torch.cuda.max_memory_allocated() / (1024**3), 3),
        }
    return None

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
mem_before = _gpu_mem()
print('GPU memory before load:', mem_before)

processor = None
model = None
load_error = None
load_seconds = None

try:
    from transformers import AutoProcessor, AutoModelForMultimodalLM
    t0 = time.perf_counter()
    processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        dtype='auto',
        device_map='auto',
        low_cpu_mem_usage=True,
        token=HF_TOKEN,
    )
    load_seconds = time.perf_counter() - t0
    print(f'Model loaded successfully in {load_seconds:.2f}s')
    print('Model class:', type(model).__name__)
    print('Processor class:', type(processor).__name__)
    try:
        print('Model device:', getattr(model, 'device', 'n/a'))
        print('HF device map:', getattr(model, 'hf_device_map', None) is not None)
    except Exception:
        pass
except Exception as e:
    load_error = e
    print('ERROR: Failed to load model/processor.')
    print('Common causes: gated access not accepted, missing/invalid HF token, incompatible transformers version, or insufficient resources.')
    print('Details:', repr(e))

mem_after = _gpu_mem()
print('GPU memory after load:', mem_after)


## E. Inference helper: `ask_gemma()`
Reusable generation helper using the official Gemma 4 chat template with timing, token counts, and graceful error handling.


In [ ]:
import time
import traceback
from typing import Any, Dict, Optional

def _move_inputs_to_model(inputs, model):
    # With device_map='auto', moving to model.device is generally appropriate for single-input inference.
    try:
        return inputs.to(model.device)
    except Exception:
        pass
    moved = {}
    for k, v in inputs.items():
        if hasattr(v, 'to'):
            try:
                moved[k] = v.to(model.device)
                continue
            except Exception:
                pass
        moved[k] = v
    return moved


def _parse_model_response(processor, raw_text: str, inputs) -> str:
    # Try official parsing API (signature may vary across versions/examples).
    try:
        parsed = processor.parse_response(raw_text, prefix=inputs.get('input_ids'))
    except TypeError:
        try:
            parsed = processor.parse_response(raw_text)
        except Exception:
            return raw_text.strip()
    except Exception:
        return raw_text.strip()

    if isinstance(parsed, dict):
        # Common keys across chat/assistant parsing utilities
        for key in ('content', 'text', 'answer'):
            val = parsed.get(key)
            if isinstance(val, str) and val.strip():
                return val.strip()
        # Fallback: join string fields except role/thinking metadata if present
        parts = []
        for k, v in parsed.items():
            if k in ('role', 'thinking', 'thought'):
                continue
            if isinstance(v, str) and v.strip():
                parts.append(v.strip())
        if parts:
            return '\n'.join(parts).strip()
    if isinstance(parsed, str):
        return parsed.strip()
    return raw_text.strip()


def ask_gemma(
    prompt: str,
    *,
    system_prompt: str = 'You are a helpful software engineering assistant.',
    max_new_tokens: int = 1024,
    deterministic: bool = True,
    enable_thinking: bool = False,
    temperature: float = 1.0,
    top_p: float = 0.95,
    top_k: int = 64,
    **generate_kwargs,
) -> Dict[str, Any]:
    """Run a single Gemma 4 chat-style generation and return response + metrics.

    Returns keys: response, raw_response, duration_seconds, token_counts, error, config
    """
    result: Dict[str, Any] = {
        'response': None,
        'raw_response': None,
        'duration_seconds': None,
        'token_counts': None,
        'error': None,
        'config': {
            'max_new_tokens': max_new_tokens,
            'deterministic': deterministic,
            'enable_thinking': enable_thinking,
            'temperature': temperature,
            'top_p': top_p,
            'top_k': top_k,
        },
    }

    if model is None or processor is None:
        result['error'] = 'Model/processor not loaded.'
        return result

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': prompt},
    ]

    try:
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            return_dict=True,
            return_tensors='pt',
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
        inputs = _move_inputs_to_model(inputs, model)
        input_len = int(inputs['input_ids'].shape[-1])

        gen_args = dict(max_new_tokens=max_new_tokens)
        if deterministic:
            gen_args.update({'do_sample': False})
        else:
            gen_args.update({'do_sample': True, 'temperature': temperature, 'top_p': top_p, 'top_k': top_k})
        gen_args.update(generate_kwargs)

        start = time.perf_counter()
        with torch.inference_mode():
            outputs = model.generate(**inputs, **gen_args)
        duration = time.perf_counter() - start

        output_ids = outputs[0]
        gen_ids = output_ids[input_len:]
        raw_text = processor.decode(gen_ids, skip_special_tokens=False)
        clean_text = _parse_model_response(processor, raw_text, inputs)

        result['response'] = clean_text
        result['raw_response'] = raw_text
        result['duration_seconds'] = round(duration, 4)
        result['token_counts'] = {
            'input_tokens': input_len,
            'generated_tokens': int(gen_ids.shape[-1]),
            'total_tokens': int(output_ids.shape[-1]),
        }
        return result

    except torch.cuda.OutOfMemoryError as e:
        if torch.cuda.is_available():
            try:
                torch.cuda.empty_cache()
            except Exception:
                pass
        result['error'] = f'CUDA OOM: {e}'
        result['traceback'] = traceback.format_exc(limit=5)
        return result
    except RuntimeError as e:
        msg = str(e)
        if 'out of memory' in msg.lower():
            if torch.cuda.is_available():
                try:
                    torch.cuda.empty_cache()
                except Exception:
                    pass
            result['error'] = f'Runtime OOM: {e}'
        elif 'context' in msg.lower() or 'length' in msg.lower():
            result['error'] = f'Possible context/length error: {e}'
        else:
            result['error'] = f'RuntimeError: {e}'
        result['traceback'] = traceback.format_exc(limit=5)
        return result
    except Exception as e:
        result['error'] = f'{type(e).__name__}: {e}'
        result['traceback'] = traceback.format_exc(limit=5)
        return result


In [ ]:
# Quick smoke test (runs only if model is loaded). Safe to skip/fail gracefully.
if model is not None and processor is not None:
    smoke = ask_gemma('Reply with exactly: READY', max_new_tokens=32, deterministic=True, enable_thinking=False)
    print('Smoke test error:', smoke.get('error'))
    print('Smoke test response:', smoke.get('response'))
    print('Smoke test metrics:', smoke.get('duration_seconds'), smoke.get('token_counts'))
else:
    print('Model not loaded; skipping smoke test.')


## F. Baseline benchmark (5 software engineering tasks)
Loads tasks from `benchmarks/coding_tasks.jsonl` and runs deterministic baseline generations.


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

def resolve_tasks_path(project_root: Path) -> Path:
    candidates = [
        project_root / 'benchmarks' / 'coding_tasks.jsonl',
        Path.cwd() / 'benchmarks' / 'coding_tasks.jsonl',
        Path('/content/benchmarks/coding_tasks.jsonl'),
        Path('/content/GemmaCode-E2B/benchmarks/coding_tasks.jsonl'),
        Path('/content/drive/MyDrive/GemmaCode-E2B/benchmarks/coding_tasks.jsonl'),
    ]
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(
        'Could not find benchmarks/coding_tasks.jsonl. '
        'Upload/clone the FULL GemmaCode-E2B repository to Colab (not just the notebook), '
        'then set PROJECT_ROOT to that repo folder and re-run from the top. '
        'Example: PROJECT_ROOT = Path(\'/content/GemmaCode-E2B\'); os.chdir(PROJECT_ROOT)'
    )

TASKS_PATH = resolve_tasks_path(PROJECT_ROOT)
print('Tasks file:', TASKS_PATH)

tasks = []
with TASKS_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            tasks.append(json.loads(line))

print(f'Loaded {len(tasks)} tasks')
for t in tasks:
    print('-', t.get('task_id'), '|', t.get('category'))


In [ ]:
# Benchmark configuration (deterministic baseline)
BENCHMARK_CONFIG = {
    'max_new_tokens': 1024,
    'deterministic': True,
    'enable_thinking': False,
    'temperature': 1.0,
    'top_p': 0.95,
    'top_k': 64,
    'system_prompt': 'You are a helpful software engineering assistant.',
}

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_timestamp = datetime.now(timezone.utc).isoformat()
print('Run ID:', run_id)

benchmark_records = []

if model is None or processor is None:
    print('Model not loaded; benchmark execution skipped. Load the model successfully in Section D before running this cell.')
else:
    for i, task in enumerate(tasks, start=1):
        print(f"\n[{i}/{len(tasks)}] Running {task['task_id']} ({task['category']})...")
        out = ask_gemma(
            task['prompt'],
            system_prompt=BENCHMARK_CONFIG['system_prompt'],
            max_new_tokens=BENCHMARK_CONFIG['max_new_tokens'],
            deterministic=BENCHMARK_CONFIG['deterministic'],
            enable_thinking=BENCHMARK_CONFIG['enable_thinking'],
            temperature=BENCHMARK_CONFIG['temperature'],
            top_p=BENCHMARK_CONFIG['top_p'],
            top_k=BENCHMARK_CONFIG['top_k'],
        )
        record = {
            'task_id': task.get('task_id'),
            'category': task.get('category'),
            'prompt': task.get('prompt'),
            'model_identifier': MODEL_ID,
            'generated_response': out.get('response'),
            'raw_response': out.get('raw_response'),
            'generation_duration_seconds': out.get('duration_seconds'),
            'token_counts': out.get('token_counts'),
            'configuration': out.get('config'),
            'error': out.get('error'),
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'run_id': run_id,
        }
        benchmark_records.append(record)
        print('  duration_s:', record['generation_duration_seconds'])
        print('  tokens:', record['token_counts'])
        print('  error:', record['error'])

    print(f"\nCompleted {len(benchmark_records)} benchmark generations.")


## G. Save results (local + Google Drive)
Saves benchmark outputs to `benchmarks/results/` and mirrors to Google Drive under `MyDrive/GemmaCode-E2B/benchmarks/results/`.
Filenames include a run ID to avoid overwriting previous benchmark results.


In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path
from datetime import datetime, timezone

# Ensure PROJECT_ROOT points to repo root when possible
if 'PROJECT_ROOT' not in globals() or not (Path(PROJECT_ROOT) / 'benchmarks').exists():
    try:
        PROJECT_ROOT = find_project_root()  # defined in Section A
        os.chdir(PROJECT_ROOT)
    except Exception:
        PROJECT_ROOT = Path.cwd().resolve()

local_results_dir = Path(PROJECT_ROOT) / 'benchmarks' / 'results'
local_results_dir.mkdir(parents=True, exist_ok=True)

if 'run_id' not in globals():
    run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
if 'run_timestamp' not in globals():
    run_timestamp = datetime.now(timezone.utc).isoformat()
if 'benchmark_records' not in globals():
    benchmark_records = []
if 'BENCHMARK_CONFIG' not in globals():
    BENCHMARK_CONFIG = {}

payload = {
    'benchmark_name': 'baseline_original',
    'model_identifier': 'google/gemma-4-E2B-it',
    'run_id': run_id,
    'created_at': run_timestamp,
    'task_file': 'benchmarks/coding_tasks.jsonl',
    'benchmark_config': BENCHMARK_CONFIG,
    'environment_summary': {
        'python_version': sys.version.split()[0],
        'torch_version': getattr(torch, '__version__', None) if 'torch' in globals() else None,
        'cuda_available': torch.cuda.is_available() if 'torch' in globals() else None,
    },
    'records': benchmark_records,
}

local_filename = f'baseline_original_{run_id}.json'
local_path = local_results_dir / local_filename
with local_path.open('w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print('Saved local results:', local_path)

# Save a local latest copy for convenience (timestamped run file remains the non-overwritten record)
latest_path = local_results_dir / 'baseline_original.json'
with latest_path.open('w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)
print('Updated local latest copy:', latest_path)

# Mount Google Drive and copy results (Colab)
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        drive_dir = Path('/content/drive/MyDrive/GemmaCode-E2B/benchmarks/results')
        drive_dir.mkdir(parents=True, exist_ok=True)
        drive_path = drive_dir / local_filename
        shutil.copy2(local_path, drive_path)
        print('Saved Drive results:', drive_path)
    except Exception as e:
        print('Google Drive save failed:', e)
        print('Local results are still saved at:', local_path)
else:
    print('Not running in Colab; skipped Google Drive mount/copy.')


## Phase 1 completion notes
- If the model loaded and all 5 tasks ran, `benchmarks/results/baseline_original_<run_id>.json` contains the baseline outputs and metrics.
- Use `src/evaluation/rubric.py` as the foundation for manual (non-fabricated) scoring across evaluation dimensions.
- Do not proceed to fine-tuning (Phase 2) until explicitly approved.
